In [24]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf


def make_charts(data, ticker, period):
    close = data["Close"].dropna()
    if close.empty:
        raise ValueError("No closing-price data was returned for this ticker.")
    df = pd.DataFrame(index=close.index)
    df["Close"] = close

    tnx_data = yf.Ticker("^TNX").history(period=period, auto_adjust=False)
    if tnx_data.empty or tnx_data["Close"].dropna().empty:
        raise ValueError("No 10-year Treasury yield data was returned.")

    
    tnx_close = (tnx_data["Close"].dropna()).reindex(close.index, method="ffill")
    df["TNX"] = tnx_close

    #Below are the 3 variable plug ins for the data used by the 3 other charts
    returns = close.pct_change().dropna() * 100
   
    volatility = close.pct_change().rolling(20, min_periods=5).std() * (252 ** 0.5) * 100

    price_fig, price_ax = plt.subplots(figsize=(8, 3.2))
    price_ax.plot(close.index, close, label="Close", color="#176B87", linewidth=1.8)
    price_ax.set_title(f"{ticker} closing price")
    price_ax.set_ylabel("Price")

    tnx_ax = price_ax.twinx()
    tnx_ax.plot(
        close.index,
        tnx_close,
        label="10-year Treasury yield",
        color="#E07A3F",
        linewidth=1.3,
    )
    tnx_ax.set_ylabel("Yield (%)")

    lines = price_ax.get_lines() + tnx_ax.get_lines()
    price_ax.legend(lines, [line.get_label() for line in lines], frameon=False)

    variables = ["Close", "TNX"]
    colors = ["#42A38A", "#E67E22"]

    volume_fig, volume_ax = plt.subplots(figsize=(8, 3.2))
    for var, color in zip(variables, colors):
        volume_ax.scatter(
        df.index,
        df[var].fillna(0),
        label=var,
        color=color,
        alpha=0.7
        )
        volume_ax.set_title(f"{ticker} trading volume")
        volume_ax.set_ylabel("Shares")
        
    


   










    returns_fig, returns_ax = plt.subplots(figsize=(8, 3.2))
    return_colors = ["#42A38A" if value >= 0 else "#D45D5D" for value in returns]
    returns_ax.bar(returns.index, returns, color=return_colors, width=1.5)
    returns_ax.axhline(0, color="#555555", linewidth=0.8)
    returns_ax.set_title(f"{ticker} daily returns")
    returns_ax.set_ylabel("Return (%)")

    volatility_fig, volatility_ax = plt.subplots(figsize=(8, 3.2))
    volatility_ax.plot(volatility.index, volatility, color="#8B6CB2", linewidth=1.6)
    volatility_ax.set_title(f"{ticker} 20-day annualized volatility")
    volatility_ax.set_ylabel("Volatility (%)")

    for axis in (price_ax, volume_ax, returns_ax, volatility_ax):
        axis.grid(axis="y", alpha=0.22)
        axis.tick_params(axis="x", labelrotation=25)

    for figure in (price_fig, volume_fig, returns_fig, volatility_fig):
        figure.tight_layout()

    return price_fig, volume_fig, returns_fig, volatility_fig


def placeholder_plot(title, message):
    figure, axis = plt.subplots(figsize=(8, 3.2))
    axis.text(0.5, 0.5, message, ha="center", va="center", color="#666666")
    axis.set_title(title)
    axis.set_axis_off()
    figure.tight_layout()
    return figure


def update_dashboard(ticker, period):
    ticker = ticker.strip().upper()
    try:
        if not ticker:
            raise ValueError("Enter a ticker symbol, such as AAPL.")
        data = yf.Ticker(ticker).history(period=period, auto_adjust=False)
        if data.empty:
            raise ValueError("No data was found. Check the ticker or try another period.")
        charts = make_charts(data, ticker, period)
        return f"Loaded {len(data):,} trading days for {ticker}.", *charts
    except Exception as error:
        message = str(error)
        charts = tuple(
            placeholder_plot(title, message)
            for title in ("Closing price", "Trading volume", "Daily returns", "Rolling volatility")
        )
        return f"Unable to load {ticker or 'ticker'}: {message}", *charts


def initial_plot(title):
    return placeholder_plot(title, "Enter a ticker and select Load charts")


demo = gr.Blocks(title="Market charts")
with demo:
    gr.Markdown("# Market charts\nExplore price history and trading activity with four Matplotlib views.")
    with gr.Row():
        ticker_input = gr.Textbox(label="Ticker", value="AAPL", placeholder="AAPL")
        period_input = gr.Dropdown(
            choices=["1mo", "3mo", "6mo", "1y", "2y", "5y"],
            value="1y",
            label="History",
        )
        load_button = gr.Button("Load charts", variant="primary")
    status = gr.Markdown("Ready. Load a ticker to display its charts.")
    with gr.Row():
        with gr.Group():
            gr.Markdown("### Price")
            price_plot = gr.Plot(value=initial_plot("Closing price"), label="Closing price")
        with gr.Group():
            gr.Markdown("### Volume")
            volume_plot = gr.Plot(value=initial_plot("Trading volume"), label="Trading volume")
    with gr.Row():
        with gr.Group():
            gr.Markdown("### Daily returns")
            returns_plot = gr.Plot(value=initial_plot("Daily returns"), label="Daily returns")
        with gr.Group():
            gr.Markdown("### Volatility")
            volatility_plot = gr.Plot(value=initial_plot("Rolling volatility"), label="Rolling volatility")

    load_button.click(
        fn=update_dashboard,
        inputs=[ticker_input, period_input],
        outputs=[status, price_plot, volume_plot, returns_plot, volatility_plot],
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
